# Equipment Utilization Baseline

This notebook runs the baseline pipeline end-to-end and exports:
- annotated.mp4
- results summary

In [1]:
# Optional: upload files from local machine if not already uploaded
# from google.colab import files
# uploaded = files.upload()
# print(list(uploaded.keys()))

import os
print('Files in /content:', sorted(os.listdir('/content')))

Files in /content: ['.config', 'old Liebherr R9350 Excavator loads a Caterpillar 777e truck at full power _ Miningstory(720P_HD).mp4', 'repo.zip', 'sample_data']


In [2]:
# Check GPU
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))

CUDA available: True
GPU count: 1
GPU name: Tesla T4


In [3]:
# Extract repo.zip -> /content/repo
import os
import zipfile
import shutil

repo_zip = '/content/repo.zip'
if not os.path.exists(repo_zip):
    zips = sorted([f for f in os.listdir('/content') if f.lower().endswith('.zip')])
    if not zips:
        raise FileNotFoundError('No .zip found in /content. Upload repo.zip first.')
    repo_zip = f"/content/{zips[0]}"

repo_dir = '/content/repo'
if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)
os.makedirs(repo_dir, exist_ok=True)

with zipfile.ZipFile(repo_zip, 'r') as zf:
    zf.extractall(repo_dir)

# Flatten one nested top-level folder if needed
children = [x for x in os.listdir(repo_dir) if x != '__MACOSX']
if len(children) == 1 and os.path.isdir(os.path.join(repo_dir, children[0])):
    inner = os.path.join(repo_dir, children[0])
    inner_items = os.listdir(inner)
    if 'colab' in inner_items or 'requirements' in inner_items:
        for name in inner_items:
            shutil.move(os.path.join(inner, name), os.path.join(repo_dir, name))
        shutil.rmtree(inner)

print('Repo extracted to:', repo_dir)
print('Repo root items:', sorted(os.listdir(repo_dir)))

Repo extracted to: /content/repo
Repo root items: ['.venv', 'colab', 'requirements']


In [4]:
# Validate required files
import os
required = [
    '/content/repo/requirements/baseline.txt',
    '/content/repo/colab/baseline_foundation/pipeline.py',
]
for p in required:
    print(p, '->', os.path.exists(p))
if not all(os.path.exists(p) for p in required):
    raise FileNotFoundError('Required project files are missing after extraction.')

/content/repo/requirements/baseline.txt -> True
/content/repo/colab/baseline_foundation/pipeline.py -> True


In [5]:
# Install dependencies
!pip -q install -r /content/repo/requirements/baseline.txt

   â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 1.2/1.2 MB 65.8 MB/s eta 0:00:00


## ACID Fine-Tuning (Recommended)
Use these cells to fine-tune YOLO on the ACID dataset (COCO format) so classes like excavator and dump_truck are learned directly by the detector.

Workflow:
1. Mount Google Drive and point to Dataset root.
2. Train YOLOv8n on ACID.
3. Automatically route best weights into the baseline pipeline run cell.

In [6]:
# Mount Drive and lock final YOLO dataset path
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print('Drive mount skipped or not available:', e)

YOLO_DRIVE_ROOT = '/content/drive/MyDrive/computer vision course/projects/Equipment Utilization & Activity Classification Prototype/yolov8'
root = Path(YOLO_DRIVE_ROOT)

print('YOLO_DRIVE_ROOT:', YOLO_DRIVE_ROOT)
if not root.exists():
    raise FileNotFoundError('YOLO dataset path not found on Drive. Check YOLO_DRIVE_ROOT.')

# Lock path for subsequent cells.
os.environ['DRIVE_YOLO_DATASET_ROOT'] = str(root)

print('data.yaml exists ->', (root / 'data.yaml').exists())
print('train exists ->', (root / 'train').exists())
print('valid exists ->', (root / 'valid').exists())
print('test exists ->', (root / 'test').exists())

Mounted at /content/drive
YOLO_DRIVE_ROOT: /content/drive/MyDrive/computer vision course/projects/Equipment Utilization & Activity Classification Prototype/yolov8
data.yaml exists -> True
train exists -> True
valid exists -> True
test exists -> True


In [7]:
# Confirm fixed YOLO dataset path (final)
import os
from pathlib import Path

DRIVE_YOLO_DATASET_ROOT = '/content/drive/MyDrive/computer vision course/projects/Equipment Utilization & Activity Classification Prototype/yolov8'
os.environ['DRIVE_YOLO_DATASET_ROOT'] = DRIVE_YOLO_DATASET_ROOT

root = Path(DRIVE_YOLO_DATASET_ROOT)
print('Using DRIVE_YOLO_DATASET_ROOT =', DRIVE_YOLO_DATASET_ROOT)
if not root.exists():
    raise FileNotFoundError('Configured YOLO dataset path does not exist.')

Using DRIVE_YOLO_DATASET_ROOT = /content/drive/MyDrive/computer vision course/projects/Equipment Utilization & Activity Classification Prototype/yolov8


In [8]:
# Prepare YOLO dataset from the fixed Drive path (supports no-copy mode + resume workflow)
import os
import shutil
from pathlib import Path

import yaml

# Set False to skip Drive -> /content copy (faster startup, slower training IO).
COPY_TO_LOCAL = False

LOCAL_DATA_ROOT = Path('/content/acid_yolo')
LOCAL_DATA_YAML = Path('/content/acid_active_data.yaml')
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
EXPECTED_CLASSES = {
    'backhoe_loader', 'cement_truck', 'compactor', 'dozer', 'dump_truck',
    'excavator', 'grader', 'mobile_crane', 'tower_crane', 'wheel_loader',
}

SOURCE_ROOT = Path(os.environ.get('DRIVE_YOLO_DATASET_ROOT', '')).expanduser()
if not str(SOURCE_ROOT).strip() or not SOURCE_ROOT.exists():
    raise FileNotFoundError(
        'DRIVE_YOLO_DATASET_ROOT is missing or invalid. Run Cell 8 then Cell 9 first.'
    )

def split_exists(root: Path, split: str) -> bool:
    return (root / split).exists() or (root / 'images' / split).exists()

def valid_yolo_root(root: Path) -> bool:
    return (root / 'data.yaml').exists() and all(split_exists(root, s) for s in ['train', 'valid', 'test'])

def resolve_yolo_root(base: Path):
    if valid_yolo_root(base):
        return base
    for yml in base.rglob('data.yaml'):
        cand = yml.parent
        if valid_yolo_root(cand):
            return cand
    return None

def resolve_split_dir(root: Path, split: str, kind: str) -> Path:
    if kind == 'images':
        candidates = [root / split / 'images', root / 'images' / split, root / split]
    else:
        candidates = [root / split / 'labels', root / 'labels' / split, root / split]
    return next((d for d in candidates if d.exists()), candidates[0])

def count_images(d: Path) -> int:
    if not d.exists():
        return 0
    return len([p for p in d.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS])

def count_labels(d: Path) -> int:
    if not d.exists():
        return 0
    return len(list(d.rglob('*.txt')))

source_root = resolve_yolo_root(SOURCE_ROOT)
if source_root is None:
    raise FileNotFoundError(
        'No valid YOLO root found under DRIVE_YOLO_DATASET_ROOT. Expected data.yaml + train/valid/test.'
    )

if COPY_TO_LOCAL:
    if LOCAL_DATA_ROOT.exists():
        shutil.rmtree(LOCAL_DATA_ROOT)
    shutil.copytree(source_root, LOCAL_DATA_ROOT)
    active_root = LOCAL_DATA_ROOT
    print('Mode: COPY_TO_LOCAL=True (Drive -> /content copy).')
else:
    active_root = source_root
    print('Mode: COPY_TO_LOCAL=False (train directly from Drive).')

source_data_yaml = source_root / 'data.yaml'
with open(source_data_yaml, 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

data_cfg['path'] = str(active_root)
with open(LOCAL_DATA_YAML, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False, allow_unicode=False)

os.environ['ACID_DATA_YAML'] = str(LOCAL_DATA_YAML)
os.environ['ACID_ACTIVE_DATA_ROOT'] = str(active_root)

raw_names = data_cfg.get('names', [])
if isinstance(raw_names, dict):
    try:
        names = [raw_names[k] for k in sorted(raw_names, key=lambda x: int(x))]
    except Exception:
        names = [raw_names[k] for k in sorted(raw_names)]
else:
    names = list(raw_names)
names = [str(x).strip() for x in names]

print('YOLO classes:', names)
print('nc =', len(names))

actual = set(names)
missing = sorted(EXPECTED_CLASSES - actual)
extra = sorted(actual - EXPECTED_CLASSES)
print('Missing expected classes:', missing if missing else 'None')
print('Extra classes:', extra if extra else 'None')

for split in ['train', 'valid', 'test']:
    img_dir = resolve_split_dir(active_root, split, 'images')
    lbl_dir = resolve_split_dir(active_root, split, 'labels')
    print(f"{split}: images={count_images(img_dir)}, labels={count_labels(lbl_dir)}")

print('Prepared runtime data.yaml:', LOCAL_DATA_YAML)
print('ACID_DATA_YAML =', os.environ['ACID_DATA_YAML'])
print('ACID_ACTIVE_DATA_ROOT =', os.environ['ACID_ACTIVE_DATA_ROOT'])

Mode: COPY_TO_LOCAL=False (train directly from Drive).
YOLO classes: ['backhoe_loader', 'cement_truck', 'compactor', 'dozer', 'dump_truck', 'excavator', 'grader', 'mobile_crane', 'tower_crane', 'wheel_loader']
nc = 10
Missing expected classes: None
Extra classes: None
train: images=20826, labels=20826
valid: images=1985, labels=1985
test: images=990, labels=990
Prepared runtime data.yaml: /content/acid_active_data.yaml
ACID_DATA_YAML = /content/acid_active_data.yaml
ACID_ACTIVE_DATA_ROOT = /content/drive/MyDrive/computer vision course/projects/Equipment Utilization & Activity Classification Prototype/yolov8


In [11]:
# Fine-tune YOLOv8n on ACID (resumable across sessions)
import os
from pathlib import Path

import torch
from ultralytics import YOLO

ACID_TRAIN = True
ACID_RESUME_IF_AVAILABLE = True
ACID_BASE_MODEL = 'yolov8n.pt'
ACID_EPOCHS = 50
ACID_BATCH = 16
ACID_IMGSZ = 640
ACID_RUN_NAME = 'acid_yolov8n_ft2'

# Safety switch: keep False to avoid accidental epoch-1 retraining from stripped checkpoints.
ACID_ALLOW_RETRAIN_FROM_STRIPPED = False

# Keep training artifacts on Drive so you can continue tomorrow.
ACID_PROJECT = '/content/drive/MyDrive/computer vision course/projects/Equipment Utilization & Activity Classification Prototype/runs'
ACID_DATA_YAML = os.environ.get('ACID_DATA_YAML', '/content/acid_active_data.yaml')

run_dir = Path(ACID_PROJECT) / ACID_RUN_NAME
last_ckpt = run_dir / 'weights' / 'last.pt'
best_ckpt = run_dir / 'weights' / 'best.pt'
results_csv = run_dir / 'results.csv'

def is_resumable_checkpoint(ckpt_path: Path) -> bool:
    """True only if checkpoint has valid optimizer state and epoch for strict resume=True flow."""
    try:
        ckpt = torch.load(str(ckpt_path), map_location='cpu')
    except Exception:
        return False

    if not isinstance(ckpt, dict):
        return False

    optimizer = ckpt.get('optimizer', None)
    epoch = ckpt.get('epoch', None)

    # Ultralytics stripped checkpoints typically keep keys but set optimizer=None and epoch=-1.
    if optimizer is None:
        return False
    if not isinstance(epoch, (int, float)) or epoch < 0:
        return False

    # Require actual optimizer payload, not an empty placeholder.
    if isinstance(optimizer, dict):
        state = optimizer.get('state', None)
        param_groups = optimizer.get('param_groups', None)
        if not state and not param_groups:
            return False

    return True

def completed_epochs_from_results(results_path: Path) -> int:
    """Infer completed epochs from results.csv row count (excluding header)."""
    if not results_path.exists():
        return 0
    try:
        with open(results_path, 'r', encoding='utf-8') as f:
            rows = sum(1 for _ in f) - 1
        return max(0, rows)
    except Exception:
        return 0

BEST_WEIGHTS = ''
if ACID_TRAIN:
    os.makedirs(ACID_PROJECT, exist_ok=True)

    completed_epochs = completed_epochs_from_results(results_csv)
    remaining_epochs = max(0, ACID_EPOCHS - completed_epochs)

    print(f'Completed epochs from results.csv: {completed_epochs}/{ACID_EPOCHS}')

    if ACID_RESUME_IF_AVAILABLE and last_ckpt.exists() and is_resumable_checkpoint(last_ckpt):
        print('Resuming training from:', last_ckpt)
        model = YOLO(str(last_ckpt))
        model.train(resume=True)

    elif ACID_RESUME_IF_AVAILABLE and last_ckpt.exists():
        run_completed = (remaining_epochs == 0) and best_ckpt.exists()

        if run_completed and not ACID_ALLOW_RETRAIN_FROM_STRIPPED:
            print('Checkpoint found but not resumable (stripped optimizer/epoch).')
            print('Run already reached target epochs; skipping retrain and using best.pt/last.pt.')

        elif not ACID_ALLOW_RETRAIN_FROM_STRIPPED and completed_epochs == 0:
            print('Checkpoint found but not resumable (stripped optimizer/epoch).')
            print('No safe epoch history found in results.csv, so retrain is skipped by default.')
            print('Set ACID_ALLOW_RETRAIN_FROM_STRIPPED=True if you intentionally want warm-start retraining.')

        else:
            # Warm-start from last.pt with explicit ACID params.
            # If history exists, train only remaining epochs instead of restarting a full 50.
            warmstart_epochs = ACID_EPOCHS if ACID_ALLOW_RETRAIN_FROM_STRIPPED else max(1, remaining_epochs)

            print('Checkpoint found but not resumable (stripped optimizer/epoch).')
            print(f'Continuing from last.pt for {warmstart_epochs} epoch(s) using explicit ACID settings...')
            model = YOLO(str(last_ckpt))
            model.train(
                data=ACID_DATA_YAML,
                epochs=warmstart_epochs,
                imgsz=ACID_IMGSZ,
                batch=ACID_BATCH,
                device=0,
                project=ACID_PROJECT,
                name=ACID_RUN_NAME,
                exist_ok=True,
                pretrained=False,
                patience=15,
                workers=2,
                verbose=True,
            )

    else:
        print('Starting new training run...')
        model = YOLO(ACID_BASE_MODEL)
        model.train(
            data=ACID_DATA_YAML,
            epochs=ACID_EPOCHS,
            imgsz=ACID_IMGSZ,
            batch=ACID_BATCH,
            device=0,
            project=ACID_PROJECT,
            name=ACID_RUN_NAME,
            pretrained=True,
            patience=15,
            workers=2,
            verbose=True,
        )
else:
    print('ACID_TRAIN is False; training skipped.')

if best_ckpt.exists():
    BEST_WEIGHTS = str(best_ckpt)
elif last_ckpt.exists():
    BEST_WEIGHTS = str(last_ckpt)

print('ACID_DATA_YAML:', ACID_DATA_YAML)
print('Run directory:', run_dir)
print('BEST_WEIGHTS:', BEST_WEIGHTS)
print('best.pt exists:', best_ckpt.exists())
print('last.pt exists:', last_ckpt.exists())

Completed epochs from results.csv: 0/50
Checkpoint found but not resumable (stripped optimizer/epoch).
No safe epoch history found in results.csv, so retrain is skipped by default.
Set ACID_ALLOW_RETRAIN_FROM_STRIPPED=True if you intentionally want warm-start retraining.
ACID_DATA_YAML: /content/acid_active_data.yaml
Run directory: /content/drive/MyDrive/computer vision course/projects/Equipment Utilization & Activity Classification Prototype/runs/acid_yolov8n_ft2
BEST_WEIGHTS: /content/drive/MyDrive/computer vision course/projects/Equipment Utilization & Activity Classification Prototype/runs/acid_yolov8n_ft2/weights/best.pt
best.pt exists: True
last.pt exists: True


In [12]:
9# Route fine-tuned weights and ACID-only class names into baseline pipeline
import os

DEFAULT_WEIGHTS = 'yolov8x.pt'
if BEST_WEIGHTS and os.path.exists(BEST_WEIGHTS):
    os.environ['PIPELINE_WEIGHTS'] = BEST_WEIGHTS
else:
    os.environ['PIPELINE_WEIGHTS'] = DEFAULT_WEIGHTS

acid_names = [
    'backhoe_loader',
    'cement_truck',
    'compactor',
    'dozer',
    'dump_truck',
    'excavator',
    'grader',
    'mobile_crane',
    'tower_crane',
    'wheel_loader',
]

# Strict filtering: ACID classes only.
os.environ['PIPELINE_ALLOWED_CLASS_NAMES'] = ','.join(acid_names)

print('PIPELINE_WEIGHTS =', os.environ['PIPELINE_WEIGHTS'])
print('PIPELINE_ALLOWED_CLASS_NAMES =', os.environ['PIPELINE_ALLOWED_CLASS_NAMES'])

PIPELINE_WEIGHTS = /content/drive/MyDrive/computer vision course/projects/Equipment Utilization & Activity Classification Prototype/runs/acid_yolov8n_ft2/weights/best.pt
PIPELINE_ALLOWED_CLASS_NAMES = backhoe_loader,cement_truck,compactor,dozer,dump_truck,excavator,grader,mobile_crane,tower_crane,wheel_loader


In [27]:
# Locate video (supports both separate upload and video inside repo.zip)
import os
import glob
import shutil

VIDEO_OVERRIDE = '/content/trimmed_3min.mp4'  # optional absolute path if you want to force a specific video
video_ext = ('*.mp4', '*.mov', '*.avi', '*.mkv', '*.MP4', '*.MOV', '*.AVI', '*.MKV')

def list_videos(base):
    found = []
    for ext in video_ext:
        found.extend(glob.glob(os.path.join(base, ext)))
    return sorted(found)

def pick_latest(paths):
    if not paths:
        return None
    return max(paths, key=lambda p: os.path.getmtime(p))

candidates = []
if VIDEO_OVERRIDE and os.path.exists(VIDEO_OVERRIDE):
    candidates = [VIDEO_OVERRIDE]
else:
    # First priority: videos uploaded directly to /content
    candidates = list_videos('/content')

    # Fallback: search recursively inside repo
    if not candidates:
        for ext in ['mp4', 'mov', 'avi', 'mkv', 'MP4', 'MOV', 'AVI', 'MKV']:
            candidates.extend(glob.glob(f'/content/repo/**/*.{ext}', recursive=True))
        candidates = sorted(set(candidates))

if not candidates:
    raise FileNotFoundError('No video found. Upload a video to /content or include it in repo.zip.')

chosen = candidates[0] if VIDEO_OVERRIDE else pick_latest(candidates)
INPUT_VIDEO = '/content/input.mp4'
if os.path.abspath(chosen) != os.path.abspath(INPUT_VIDEO):
    if os.path.exists(INPUT_VIDEO):
        os.remove(INPUT_VIDEO)
    shutil.copy2(chosen, INPUT_VIDEO)

print('Video candidates:', candidates[:5], '... total=', len(candidates))
print('Selected video:', chosen)
print('Prepared input:', INPUT_VIDEO)

Video candidates: ['/content/trimmed_3min.mp4'] ... total= 1
Selected video: /content/trimmed_3min.mp4
Prepared input: /content/input.mp4


In [28]:
# Optional: build a short clip for fast tuning
import os
import glob
import shutil
import subprocess
import cv2

# IMPORTANT: Set to False to process the full uploaded video.
USE_FAST_TUNE = False
FAST_TUNE_SECONDS = 120  # first 2 minutes when fast tuning is enabled

# Recover INPUT_VIDEO automatically if previous setup cells were skipped.
video_ext = ('*.mp4', '*.mov', '*.avi', '*.mkv', '*.MP4', '*.MOV', '*.AVI', '*.MKV')

def list_videos(base):
    found = []
    for ext in video_ext:
        found.extend(glob.glob(os.path.join(base, ext)))
    return sorted(found)

def pick_latest(paths):
    if not paths:
        return None
    return max(paths, key=lambda p: os.path.getmtime(p))

def video_has_frames(path):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        cap.release()
        return False
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0
    ok, _ = cap.read()
    cap.release()
    return bool(ok) and frame_count > 0

INPUT_VIDEO = globals().get('INPUT_VIDEO', '/content/input.mp4')
if not os.path.exists(INPUT_VIDEO):
    candidates = list_videos('/content')
    if not candidates:
        for ext in ['mp4', 'mov', 'avi', 'mkv', 'MP4', 'MOV', 'AVI', 'MKV']:
            candidates.extend(glob.glob(f'/content/repo/**/*.{ext}', recursive=True))
        candidates = sorted(set(candidates))

    chosen = pick_latest(candidates)
    if not chosen:
        raise FileNotFoundError(
            'No video found. Run the video setup cell first or upload a video to /content.'
        )

    if os.path.abspath(chosen) != os.path.abspath(INPUT_VIDEO):
        if os.path.exists(INPUT_VIDEO):
            os.remove(INPUT_VIDEO)
        shutil.copy2(chosen, INPUT_VIDEO)
    print('Recovered INPUT_VIDEO from:', chosen)

RUN_VIDEO = INPUT_VIDEO
cap_info = cv2.VideoCapture(INPUT_VIDEO)
if not cap_info.isOpened():
    raise RuntimeError(
        f'Cannot open video: {INPUT_VIDEO}. Re-run the video setup cell or upload a supported video.'
    )
fps_info = cap_info.get(cv2.CAP_PROP_FPS) or 25.0
frames_info = cap_info.get(cv2.CAP_PROP_FRAME_COUNT) or 0
cap_info.release()
input_duration = frames_info / max(1e-6, fps_info)
print(f'INPUT duration (sec): {round(input_duration, 2)}')

if USE_FAST_TUNE:
    fast_path = '/content/input_fast_tune.mp4'
    if os.path.exists(fast_path):
        os.remove(fast_path)

    if input_duration <= FAST_TUNE_SECONDS + 1:
        RUN_VIDEO = INPUT_VIDEO
        print('FAST_TUNE window >= input duration, using INPUT_VIDEO directly.')
    else:
        # First try fast stream-copy clip.
        copy_cmd = [
            'ffmpeg', '-y', '-hide_banner', '-loglevel', 'error',
            '-i', INPUT_VIDEO,
            '-t', str(int(FAST_TUNE_SECONDS)),
            '-c', 'copy',
            fast_path,
        ]
        try:
            copy_rc = subprocess.run(copy_cmd, check=False).returncode
        except FileNotFoundError:
            copy_rc = 1

        if copy_rc != 0 or not os.path.exists(fast_path) or not video_has_frames(fast_path):
            # Fallback: re-encode to ensure decodable output.
            reenc_cmd = [
                'ffmpeg', '-y', '-hide_banner', '-loglevel', 'error',
                '-i', INPUT_VIDEO,
                '-t', str(int(FAST_TUNE_SECONDS)),
                '-an',
                '-c:v', 'libx264',
                '-preset', 'veryfast',
                '-crf', '23',
                fast_path,
            ]
            try:
                reenc_rc = subprocess.run(reenc_cmd, check=False).returncode
            except FileNotFoundError:
                reenc_rc = 1

            if reenc_rc != 0 or not os.path.exists(fast_path) or not video_has_frames(fast_path):
                RUN_VIDEO = INPUT_VIDEO
                print('FAST_TUNE clip creation failed; falling back to INPUT_VIDEO.')
            else:
                RUN_VIDEO = fast_path
                print('FAST_TUNE enabled: created decodable clip via ffmpeg re-encode.')
        else:
            RUN_VIDEO = fast_path
            print('FAST_TUNE enabled: created clip via ffmpeg stream copy.')

    if RUN_VIDEO == fast_path:
        cap_fast = cv2.VideoCapture(RUN_VIDEO)
        fps_fast = cap_fast.get(cv2.CAP_PROP_FPS) or 25.0
        frames_fast = cap_fast.get(cv2.CAP_PROP_FRAME_COUNT) or 0
        cap_fast.release()
        print(f'FAST_TUNE duration (sec): {round(frames_fast / max(1e-6, fps_fast), 2)}')
else:
    print('FAST_TUNE disabled: processing full INPUT_VIDEO.')

print('RUN_VIDEO:', RUN_VIDEO)

INPUT duration (sec): 174.02
FAST_TUNE disabled: processing full INPUT_VIDEO.
RUN_VIDEO: /content/input.mp4


In [29]:
# Run baseline pipeline
import json
import os
import subprocess
import time
import cv2

# Fallback defaults so this cell works even if previous setup cells were skipped.
REPO_DIR = globals().get("REPO_DIR", "/content/repo")
OUTPUT_DIR = globals().get("OUTPUT_DIR", "/content/outputs_baseline")
INPUT_VIDEO = globals().get("INPUT_VIDEO", "/content/input.mp4")
RUN_VIDEO = globals().get("RUN_VIDEO", INPUT_VIDEO)

os.makedirs(OUTPUT_DIR, exist_ok=True)

def video_has_frames(path):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        cap.release()
        return False
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0
    ok, _ = cap.read()
    cap.release()
    return bool(ok) and frame_count > 0

run_video = RUN_VIDEO if os.path.exists(RUN_VIDEO) else INPUT_VIDEO
if not video_has_frames(run_video):
    print(f"Warning: RUN_VIDEO is invalid or empty ({run_video}). Falling back to INPUT_VIDEO.")
    run_video = INPUT_VIDEO
if not video_has_frames(run_video):
    raise FileNotFoundError(f"No valid readable video found. INPUT_VIDEO={INPUT_VIDEO}, RUN_VIDEO={RUN_VIDEO}")

pipeline_path = f"{REPO_DIR}/colab/baseline_foundation/pipeline.py"
if not os.path.exists(pipeline_path):
    raise FileNotFoundError(f"Pipeline not found: {pipeline_path}")

# Sanity check to avoid stale repo confusion.
required_flags = [
    "--full_motion_thr",
    "--micro_motion_ratio",
    "--state_inactive_votes",
    "--dup_vertical_x_overlap_thr",
    "--new_id_min_conf",
    "--min_entity_age_s",
    "--motion_border_margin",
    "--min_motion_occupancy",
    "--min_part_motion_occupancy",
    "--active_hold_s",
    "--sticky_detach_min_gap_s",
]
with open(pipeline_path, "r", encoding="utf-8") as f:
    src = f.read()
for flag in required_flags:
    if flag not in src:
        raise RuntimeError(f"Pipeline seems stale, missing {flag}. Re-upload latest repo.zip.")

# Keep expected machine count configurable per video.
# Default is 12 for this site; override with os.environ["EXPECTED_MACHINES"].
EXPECTED_MACHINES = max(12, int(os.environ.get("EXPECTED_MACHINES", "12")))
max_eq_per_frame = max(4, EXPECTED_MACHINES)

# Dynamic detector weights/classes (from ACID fine-tune cells when available).
PIPELINE_WEIGHTS = os.environ.get("PIPELINE_WEIGHTS", "yolov8x.pt")
if PIPELINE_WEIGHTS != "yolov8x.pt" and not os.path.exists(PIPELINE_WEIGHTS):
    print(f"Warning: PIPELINE_WEIGHTS not found ({PIPELINE_WEIGHTS}), falling back to yolov8x.pt")
    PIPELINE_WEIGHTS = "yolov8x.pt"

default_allowed = [
    "backhoe_loader",
    "cement_truck",
    "compactor",
    "dozer",
    "dump_truck",
    "excavator",
    "grader",
    "mobile_crane",
    "tower_crane",
    "wheel_loader",
    "truck",
    "dump truck",
    "loader",
    "backhoe",
    "bulldozer",
]
allowed_raw = os.environ.get("PIPELINE_ALLOWED_CLASS_NAMES", "")
if allowed_raw.strip():
    allowed_names = [x.strip() for x in allowed_raw.split(",") if x.strip()]
else:
    allowed_names = default_allowed

is_finetuned = PIPELINE_WEIGHTS not in {"yolov8x.pt", "yolov8n.pt", "yolov8s.pt", "yolov8m.pt", "yolov8l.pt"}
conf_thr = os.environ.get("PIPELINE_CONF", "0.30" if is_finetuned else "0.13")
min_box_area = os.environ.get("PIPELINE_MIN_BOX_AREA", "300" if is_finetuned else "350")

# Show actual run video duration to avoid confusion about clip/full processing.
meta_cap = cv2.VideoCapture(run_video)
meta_fps = meta_cap.get(cv2.CAP_PROP_FPS) or 25.0
meta_frames = meta_cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0
meta_cap.release()
print(f"Run video: {run_video}")
print(f"Run video duration (sec): {round(meta_frames / max(1e-6, meta_fps), 2)}")
print(f"Expected machines for this run: {EXPECTED_MACHINES}")
print(f"Weights: {PIPELINE_WEIGHTS}")
print(f"Allowed classes ({len(allowed_names)}): {allowed_names}")
print("Using tuned preset v4: high motion sensitivity + stronger ID continuity")

cmd = [
    "python", pipeline_path,
    "--input_video", run_video,
    "--output_dir", OUTPUT_DIR,
    "--weights", PIPELINE_WEIGHTS,
    "--tracker", "bytetrack.yaml",
    "--device", "0",
    "--imgsz", "1536",
    "--max_det", "250",
    "--conf", conf_thr,
    "--iou", "0.50",
    "--process_every_n_frames", "1",
    "--max_equipment_per_frame", str(max_eq_per_frame),
    "--min_track_hits", "6",
    "--min_entity_age_s", "0.35",
    "--min_box_area", min_box_area,
    "--motion_smooth_window", "9",
    "--full_motion_thr", "0.56",
    "--arm_motion_thr", "0.42",
    "--base_motion_thr", "0.74",
    "--center_motion_thr", "0.14",
    "--micro_motion_ratio", "0.46",
    "--global_motion_scale", "0.88",
    "--motion_border_margin", "0.20",
    "--min_motion_occupancy", "0.06",
    "--min_part_motion_occupancy", "0.03",
    "--state_smooth_window", "9",
    "--state_active_votes", "2",
    "--state_inactive_votes", "8",
    "--dup_iou_thr", "0.36",
    "--dup_containment_thr", "0.80",
    "--dup_x_overlap_thr", "0.68",
    "--dup_y_overlap_thr", "0.34",
    "--dup_center_dist_ratio", "0.54",
    "--dup_max_area_ratio", "3.8",
    "--dup_vertical_x_overlap_thr", "0.80",
    "--dup_vertical_gap_ratio", "0.60",
    "--dup_vertical_center_x_ratio", "0.30",
    "--dup_vertical_max_area_ratio", "5.0",
    "--track_id_ttl_s", "10.0",
    "--new_id_min_conf", "0.55",
    "--equipment_owner_hold_s", "8.0",
    "--active_hold_s", "1.00",
    "--reid_max_gap_s", "12.0",
    "--reid_memory_s", "60.0",
    "--reid_min_iou", "0.02",
    "--reid_min_app", "0.56",
    "--reid_max_center_dist_ratio", "0.33",
    "--reid_max_area_ratio", "4.5",
    "--reid_max_aspect_ratio_ratio", "2.6",
    "--reid_long_gap_center_ratio", "0.12",
    "--reid_min_match_score", "0.28",
    "--reid_iou_weight", "0.24",
    "--reid_app_weight", "0.50",
    "--reid_center_weight", "0.26",
    "--reid_hist_bins", "16",
    "--sticky_min_iou", "0.02",
    "--sticky_min_app", "0.56",
    "--sticky_max_area_ratio", "4.0",
    "--sticky_max_aspect_ratio_ratio", "2.3",
    "--sticky_max_center_dist_ratio", "0.30",
    "--sticky_detach_min_gap_s", "2.20",
    "--summary_min_tracked_seconds", "1.0",
    "--allowed_class_names", *allowed_names,
]

print("\nRunning command:\n", " ".join(cmd))
start_t = time.time()
proc = subprocess.run(cmd, capture_output=True, text=True)
elapsed_s = time.time() - start_t

meta = {
    "elapsed_seconds": round(elapsed_s, 2),
    "elapsed_minutes": round(elapsed_s / 60.0, 2),
    "device": "0",
    "run_video": run_video,
    "command": cmd,
}
with open(f"{OUTPUT_DIR}/run_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

if proc.returncode != 0:
    print(proc.stdout[-4000:])
    print(proc.stderr[-4000:])
    with open(f"{OUTPUT_DIR}/last_error.log", "w", encoding="utf-8") as f:
        f.write((proc.stdout or "") + "\n" + (proc.stderr or ""))
    raise RuntimeError("Pipeline failed. See output and last_error.log")

print("Pipeline run finished.")
print("Run metadata:")
print(meta)
if proc.stdout:
    print(proc.stdout[-1200:])

Run video: /content/input.mp4
Run video duration (sec): 174.02
Expected machines for this run: 12
Weights: /content/drive/MyDrive/computer vision course/projects/Equipment Utilization & Activity Classification Prototype/runs/acid_yolov8n_ft2/weights/best.pt
Allowed classes (10): ['backhoe_loader', 'cement_truck', 'compactor', 'dozer', 'dump_truck', 'excavator', 'grader', 'mobile_crane', 'tower_crane', 'wheel_loader']
Using tuned preset v4: high motion sensitivity + stronger ID continuity

Running command:
 python /content/repo/colab/baseline_foundation/pipeline.py --input_video /content/input.mp4 --output_dir /content/outputs_baseline --weights /content/drive/MyDrive/computer vision course/projects/Equipment Utilization & Activity Classification Prototype/runs/acid_yolov8n_ft2/weights/best.pt --tracker bytetrack.yaml --device 0 --imgsz 1536 --max_det 250 --conf 0.30 --iou 0.50 --process_every_n_frames 1 --max_equipment_per_frame 12 --min_track_hits 6 --min_entity_age_s 0.35 --min_box_a

In [30]:
# Inspect outputs
import os
import json
import pandas as pd

out = '/content/outputs_baseline'
print('Output files:', sorted(os.listdir(out)))

run_meta_path = f'{out}/run_meta.json'
if os.path.exists(run_meta_path):
    print('\nRun metadata:')
    with open(run_meta_path, 'r', encoding='utf-8') as f:
        print(json.load(f))

print('\nFirst 5 events:')
with open(f'{out}/events.jsonl', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(json.loads(line))

print('\nSummary:')
summary = pd.read_csv(f'{out}/summary.csv')
print(summary)
print('\nSummary rows:', len(summary))

Output files: ['annotated.mp4', 'events.jsonl', 'run_meta.json', 'summary.csv']

Run metadata:
{'elapsed_seconds': 1511.35, 'elapsed_minutes': 25.19, 'device': '0', 'run_video': '/content/input.mp4', 'command': ['python', '/content/repo/colab/baseline_foundation/pipeline.py', '--input_video', '/content/input.mp4', '--output_dir', '/content/outputs_baseline', '--weights', '/content/drive/MyDrive/computer vision course/projects/Equipment Utilization & Activity Classification Prototype/runs/acid_yolov8n_ft2/weights/best.pt', '--tracker', 'bytetrack.yaml', '--device', '0', '--imgsz', '1536', '--max_det', '250', '--conf', '0.30', '--iou', '0.50', '--process_every_n_frames', '1', '--max_equipment_per_frame', '12', '--min_track_hits', '6', '--min_entity_age_s', '0.35', '--min_box_area', '300', '--motion_smooth_window', '9', '--full_motion_thr', '0.56', '--arm_motion_thr', '0.42', '--base_motion_thr', '0.74', '--center_motion_thr', '0.14', '--micro_motion_ratio', '0.46', '--global_motion_scale

In [31]:
# Quality snapshot for tuning
import json
import os
import pandas as pd

OUT = OUTPUT_DIR
events_path = f"{OUT}/events.jsonl"
summary_path = f"{OUT}/summary.csv"

if not os.path.exists(events_path):
    raise FileNotFoundError(f"Missing events file: {events_path}")
if not os.path.exists(summary_path):
    raise FileNotFoundError(f"Missing summary file: {summary_path}")

rows = []
with open(events_path, "r", encoding="utf-8") as f:
    for line in f:
        e = json.loads(line)
        rows.append({
            "frame_id": e.get("frame_id"),
            "track_id": e.get("track_id"),
            "equipment_id": e.get("equipment_id"),
            "equipment_class": e.get("equipment_class"),
            "state": e.get("utilization", {}).get("current_state"),
            "activity": e.get("utilization", {}).get("current_activity"),
            "motion_source": e.get("utilization", {}).get("motion_source"),
            "tracked_seconds": e.get("time_analytics", {}).get("total_tracked_seconds", 0.0),
            "total_idle_seconds": e.get("time_analytics", {}).get("total_idle_seconds", 0.0),
            "utilization_percent": e.get("time_analytics", {}).get("utilization_percent", 0.0),
            "current_dwell_seconds": e.get("time_analytics", {}).get("current_dwell_seconds", 0.0),
        })

summary_df = pd.read_csv(summary_path)
df = pd.DataFrame(rows)

print("Summary:")
print(summary_df)
print("\nSummary rows:", len(summary_df))

print("\nTotal events:", len(df))
print("Unique equipment_ids:", df["equipment_id"].nunique())

# Set expected count by environment variable when needed.
# Default for this site is 12; override when running a different scene.
expected_machines = int(os.environ.get("EXPECTED_MACHINES", "12"))
print("Expected machines:", expected_machines)
print("ID excess over expected:", max(0, df["equipment_id"].nunique() - expected_machines))

active_waiting = ((df["state"] == "ACTIVE") & (df["activity"] == "WAITING")).sum()
active_none = ((df["state"] == "ACTIVE") & (df["motion_source"] == "none")).sum()
print("ACTIVE+WAITING inconsistencies:", int(active_waiting))
print("ACTIVE+none-motion anomalies:", int(active_none))
print("Note: current_dwell_seconds is current idle streak; total_idle_seconds is cumulative idle.")

print("\nTop equipment IDs by event count:")
print(df["equipment_id"].value_counts().head(10))

print("\nState counts:")
print(df["state"].value_counts(dropna=False))

print("\nMotion source counts:")
print(df["motion_source"].value_counts(dropna=False))

print("\nActivity counts:")
print(df["activity"].value_counts(dropna=False))

print("\nTrack IDs per equipment_id (top 10):")
if "track_id" in df.columns:
    print(df.groupby("equipment_id")["track_id"].nunique().sort_values(ascending=False).head(10))

print("\nEquipment IDs per track_id (top 10):")
if "track_id" in df.columns:
    print(df.groupby("track_id")["equipment_id"].nunique().sort_values(ascending=False).head(10))

print("\nSample arm_only rows:")
cols = [
    "frame_id", "track_id", "equipment_id", "equipment_class", "state", "activity",
    "motion_source", "tracked_seconds", "total_idle_seconds", "utilization_percent", "current_dwell_seconds",
]
print(df[df["motion_source"] == "arm_only"][cols].head(10))

print("\nLast metrics per equipment (top tracked):")
last = df.sort_values("frame_id").groupby("equipment_id").tail(1)
last = last.sort_values("tracked_seconds", ascending=False).head(10)
print(last[[
    "frame_id", "track_id", "equipment_id", "equipment_class", "state", "activity", "motion_source",
    "tracked_seconds", "total_idle_seconds", "utilization_percent", "current_dwell_seconds",
]])

Summary:
  equipment_id equipment_class  total_tracked_seconds  total_active_seconds  \
0      EQ-0001       excavator                118.135                83.500   
1      EQ-0002           dozer                173.657                95.812   
2      EQ-0003       excavator                126.893                35.335   
3      EQ-0004      dump_truck                 61.595                61.578   
4      EQ-0005       excavator                 67.634                22.990   
5      EQ-0006      dump_truck                 52.119                32.816   
6      EQ-0008       excavator                 34.268                29.046   

   total_idle_seconds  utilization_percent  
0              34.635                70.68  
1              77.844                55.17  
2              91.558                27.85  
3               0.017                99.97  
4              44.645                33.99  
5              19.303                62.96  
6               5.222                84.76 

In [34]:
# build SQLite analytics DB from pipeline artifacts
import json
import os
import sqlite3
import pandas as pd

OUT = globals().get("OUTPUT_DIR", "/content/outputs_baseline")
events_path = f"{OUT}/events.jsonl"
summary_path = f"{OUT}/summary.csv"
db_path = f"{OUT}/analytics_day3.db"

if not os.path.exists(events_path):
    raise FileNotFoundError(f"Missing events file: {events_path}")
if not os.path.exists(summary_path):
    raise FileNotFoundError(f"Missing summary file: {summary_path}")

rows = []
with open(events_path, "r", encoding="utf-8") as f:
    for line in f:
        e = json.loads(line)
        util = e.get("utilization", {})
        ta = e.get("time_analytics", {})
        rows.append(
            {
                "frame_id": e.get("frame_id"),
                "track_id": e.get("track_id"),
                "equipment_id": e.get("equipment_id"),
                "equipment_class": e.get("equipment_class"),
                "timestamp": e.get("timestamp"),
                "confidence": e.get("confidence"),
                "state": util.get("current_state"),
                "activity": util.get("current_activity"),
                "motion_source": util.get("motion_source"),
                "total_tracked_seconds": ta.get("total_tracked_seconds", 0.0),
                "total_active_seconds": ta.get("total_active_seconds", 0.0),
                "total_idle_seconds": ta.get("total_idle_seconds", 0.0),
                "utilization_percent": ta.get("utilization_percent", 0.0),
                "current_dwell_seconds": ta.get("current_dwell_seconds", 0.0),
            }
        )

events_df = pd.DataFrame(rows)
if events_df.empty:
    raise RuntimeError("events.jsonl is empty. Run the pipeline cell first.")

summary_df = pd.read_csv(summary_path)

conn = sqlite3.connect(db_path)
events_df.to_sql("events", conn, if_exists="replace", index=False)
summary_df.to_sql("summary", conn, if_exists="replace", index=False)

conn.execute("CREATE INDEX IF NOT EXISTS idx_events_equipment_frame ON events(equipment_id, frame_id)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_events_track ON events(track_id)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_events_state ON events(state)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_events_activity ON events(activity)")
conn.commit()

table_counts = pd.read_sql_query(
    """
    SELECT 'events' AS table_name, COUNT(*) AS row_count FROM events
    UNION ALL
    SELECT 'summary' AS table_name, COUNT(*) AS row_count FROM summary
    """,
    conn,
)
conn.close()

os.environ["ANALYTICS_DB"] = db_path
print("ANALYTICS_DB:", db_path)
print("Loaded tables:")
print(table_counts)

ANALYTICS_DB: /content/outputs_baseline/analytics_day3.db
Loaded tables:
  table_name  row_count
0     events      16228
1    summary          7


In [35]:
# query analytics KPIs from SQLite DB
import os
import sqlite3
import pandas as pd

OUT = globals().get("OUTPUT_DIR", "/content/outputs_baseline")
db_path = os.environ.get("ANALYTICS_DB", f"{OUT}/analytics_day3.db")
if not os.path.exists(db_path):
    raise FileNotFoundError(f"Analytics DB not found: {db_path}. Run the previous Day3 cell first.")

conn = sqlite3.connect(db_path)

def sql_df(query: str) -> pd.DataFrame:
    return pd.read_sql_query(query, conn)

print("KPI 1) Fleet utilization summary")
kpi1 = sql_df(
    """
    SELECT
        COUNT(DISTINCT equipment_id) AS unique_equipment_ids,
        ROUND(AVG(utilization_percent), 2) AS avg_equipment_utilization_pct,
        SUM(CASE WHEN state = 'ACTIVE' THEN 1 ELSE 0 END) AS active_frames,
        SUM(CASE WHEN state = 'INACTIVE' THEN 1 ELSE 0 END) AS inactive_frames,
        ROUND(
            100.0 * SUM(CASE WHEN state = 'ACTIVE' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0),
            2
        ) AS active_frame_ratio_pct
    FROM events
    """
)
print(kpi1.to_string(index=False))

print("\nKPI 2) Per-equipment activity mix (%)")
kpi2 = sql_df(
    """
    SELECT
        equipment_id,
        equipment_class,
        ROUND(100.0 * SUM(CASE WHEN activity = 'DIGGING' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS digging_pct,
        ROUND(100.0 * SUM(CASE WHEN activity = 'SWINGING_LOADING' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS swinging_loading_pct,
        ROUND(100.0 * SUM(CASE WHEN activity = 'PUSHING' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS pushing_pct,
        ROUND(100.0 * SUM(CASE WHEN activity = 'MOVING' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS moving_pct,
        ROUND(100.0 * SUM(CASE WHEN activity = 'DUMPING' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS dumping_pct,
        ROUND(100.0 * SUM(CASE WHEN activity = 'WAITING' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS waiting_pct
    FROM events
    GROUP BY equipment_id, equipment_class
    ORDER BY equipment_id
    """
)
print(kpi2.to_string(index=False))

print("\nKPI 3) Fragmentation audit (tracks per equipment)")
kpi3 = sql_df(
    """
    SELECT
        equipment_id,
        COUNT(DISTINCT track_id) AS tracks_per_equipment,
        MIN(frame_id) AS first_frame,
        MAX(frame_id) AS last_frame
    FROM events
    GROUP BY equipment_id
    ORDER BY tracks_per_equipment DESC, equipment_id
    """
)
print(kpi3.to_string(index=False))

print("\nKPI 4) Reused tracks (track mapped to multiple equipment IDs)")
kpi4 = sql_df(
    """
    SELECT
        track_id,
        COUNT(DISTINCT equipment_id) AS equipment_ids_on_track
    FROM events
    GROUP BY track_id
    HAVING COUNT(DISTINCT equipment_id) > 1
    ORDER BY equipment_ids_on_track DESC, track_id
    LIMIT 20
    """
)
if kpi4.empty:
    print("No reused tracks found.")
else:
    print(kpi4.to_string(index=False))

kpi1.assign(snapshot_source=os.path.basename(db_path)).to_sql(
    "kpi_snapshot", conn, if_exists="replace", index=False
)
conn.close()

print("\nSaved KPI snapshot table: kpi_snapshot")

KPI 1) Fleet utilization summary
 unique_equipment_ids  avg_equipment_utilization_pct  active_frames  inactive_frames  active_frame_ratio_pct
                    7                          69.55           9171             7057                   56.51

KPI 2) Per-equipment activity mix (%)
equipment_id equipment_class  digging_pct  swinging_loading_pct  pushing_pct  moving_pct  dumping_pct  waiting_pct
     EQ-0001       excavator        16.66                 49.91         0.00        0.00         2.36        31.07
     EQ-0002           dozer         0.00                  0.00        36.65        0.00         0.00        63.35
     EQ-0003       excavator         1.13                 47.52         0.00        0.00         0.00        51.35
     EQ-0004      dump_truck         0.00                  0.00         0.00       80.00         0.00        20.00
     EQ-0005       excavator         0.00                 50.33         0.00        0.00         0.00        49.67
     EQ-0006      du

In [36]:
# build dashboard-ready tables from analytics DB
import os
import sqlite3
import pandas as pd

OUT = globals().get("OUTPUT_DIR", "/content/outputs_baseline")
db_path = os.environ.get("ANALYTICS_DB", f"{OUT}/analytics_day3.db")
if not os.path.exists(db_path):
    raise FileNotFoundError(f"Analytics DB not found: {db_path}. Run Day3 DB build cell first.")

conn = sqlite3.connect(db_path)

activity_mix = pd.read_sql_query(
    """
    SELECT
        equipment_id,
        equipment_class,
        ROUND(100.0 * SUM(CASE WHEN activity = 'DIGGING' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS digging_pct,
        ROUND(100.0 * SUM(CASE WHEN activity = 'SWINGING_LOADING' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS swinging_loading_pct,
        ROUND(100.0 * SUM(CASE WHEN activity = 'PUSHING' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS pushing_pct,
        ROUND(100.0 * SUM(CASE WHEN activity = 'MOVING' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS moving_pct,
        ROUND(100.0 * SUM(CASE WHEN activity = 'DUMPING' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS dumping_pct,
        ROUND(100.0 * SUM(CASE WHEN activity = 'WAITING' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS waiting_pct
    FROM events
    GROUP BY equipment_id, equipment_class
    ORDER BY equipment_id
    """,
    conn,
 )

fragmentation_audit = pd.read_sql_query(
    """
    SELECT
        equipment_id,
        equipment_class,
        tracks_per_equipment,
        CASE
            WHEN tracks_per_equipment >= 10 THEN 'high'
            WHEN tracks_per_equipment >= 5 THEN 'medium'
            ELSE 'low'
        END AS fragmentation_risk
    FROM (
        SELECT
            e.equipment_id AS equipment_id,
            MIN(e.equipment_class) AS equipment_class,
            COUNT(DISTINCT e.track_id) AS tracks_per_equipment
        FROM events e
        GROUP BY e.equipment_id
    ) t
    ORDER BY tracks_per_equipment DESC, equipment_id
    """,
    conn,
 )

tracking_health = pd.read_sql_query(
    """
    WITH per_eq AS (
        SELECT equipment_id, COUNT(DISTINCT track_id) AS tracks_per_equipment
        FROM events
        GROUP BY equipment_id
    ),
    reused AS (
        SELECT COUNT(*) AS reused_tracks_count
        FROM (
            SELECT track_id
            FROM events
            GROUP BY track_id
            HAVING COUNT(DISTINCT equipment_id) > 1
        ) x
    ),
    totals AS (
        SELECT
            COUNT(DISTINCT equipment_id) AS unique_equipment_ids,
            COUNT(*) AS total_event_rows,
            ROUND(AVG(utilization_percent), 2) AS avg_frame_utilization_pct,
            ROUND(100.0 * SUM(CASE WHEN state = 'ACTIVE' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS active_frame_ratio_pct
        FROM events
    )
    SELECT
        totals.unique_equipment_ids,
        totals.total_event_rows,
        totals.avg_frame_utilization_pct,
        totals.active_frame_ratio_pct,
        ROUND(AVG(per_eq.tracks_per_equipment), 2) AS avg_tracks_per_equipment,
        MAX(per_eq.tracks_per_equipment) AS max_tracks_per_equipment,
        reused.reused_tracks_count
    FROM totals, per_eq, reused
    """,
    conn,
 )

activity_mix.to_sql("activity_mix", conn, if_exists="replace", index=False)
fragmentation_audit.to_sql("fragmentation_audit", conn, if_exists="replace", index=False)
tracking_health.to_sql("tracking_health", conn, if_exists="replace", index=False)
conn.close()

activity_mix_csv = f"{OUT}/activity_mix.csv"
fragmentation_csv = f"{OUT}/fragmentation_audit.csv"
tracking_health_csv = f"{OUT}/tracking_health.csv"
activity_mix.to_csv(activity_mix_csv, index=False)
fragmentation_audit.to_csv(fragmentation_csv, index=False)
tracking_health.to_csv(tracking_health_csv, index=False)

print("Saved DB tables: activity_mix, fragmentation_audit, tracking_health")
print("Saved CSV files:")
print(" -", activity_mix_csv)
print(" -", fragmentation_csv)
print(" -", tracking_health_csv)

print("\ntracking_health:")
print(tracking_health.to_string(index=False))

print("\nTop fragmentation rows:")
print(fragmentation_audit.head(10).to_string(index=False))

Saved DB tables: activity_mix, fragmentation_audit, tracking_health
Saved CSV files:
 - /content/outputs_baseline/activity_mix.csv
 - /content/outputs_baseline/fragmentation_audit.csv
 - /content/outputs_baseline/tracking_health.csv

tracking_health:
 unique_equipment_ids  total_event_rows  avg_frame_utilization_pct  active_frame_ratio_pct  avg_tracks_per_equipment  max_tracks_per_equipment  reused_tracks_count
                    7             16228                      69.55                   56.51                      5.71                        15                    1

Top fragmentation rows:
equipment_id equipment_class  tracks_per_equipment fragmentation_risk
     EQ-0003       excavator                    15               high
     EQ-0005       excavator                     7             medium
     EQ-0001       excavator                     6             medium
     EQ-0002           dozer                     6             medium
     EQ-0004      dump_truck                  

In [40]:
# launch interactive dashboard from analytics DB
import importlib
import os
import sqlite3
import subprocess
import sys
import pandas as pd

def ensure_package(pkg_name: str, import_name: str = None):
    mod = import_name or pkg_name
    try:
        return importlib.import_module(mod)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pkg_name])
        return importlib.import_module(mod)

plotly = ensure_package("plotly")
ipywidgets = ensure_package("ipywidgets")

import plotly.express as px
import plotly.graph_objects as go
from IPython.display import clear_output, display, Markdown
import ipywidgets as widgets

OUT = globals().get("OUTPUT_DIR", "/content/outputs_baseline")
db_path = os.environ.get("ANALYTICS_DB", f"{OUT}/analytics_day3.db")
if not os.path.exists(db_path):
    raise FileNotFoundError(f"Analytics DB not found: {db_path}. Run the Day3 DB build cell first.")

conn = sqlite3.connect(db_path)
events_df = pd.read_sql_query(
    "SELECT frame_id, equipment_id, state FROM events ORDER BY frame_id",
    conn,
 )
summary_df = pd.read_sql_query(
    "SELECT equipment_id, equipment_class, total_tracked_seconds, total_active_seconds, total_idle_seconds, utilization_percent FROM summary ORDER BY equipment_id",
    conn,
 )
activity_mix_df = pd.read_sql_query(
    "SELECT equipment_id, equipment_class, digging_pct, swinging_loading_pct, pushing_pct, moving_pct, dumping_pct, waiting_pct FROM activity_mix ORDER BY equipment_id",
    conn,
 )
fragmentation_df = pd.read_sql_query(
    "SELECT equipment_id, equipment_class, tracks_per_equipment, fragmentation_risk FROM fragmentation_audit ORDER BY tracks_per_equipment DESC, equipment_id",
    conn,
 )
tracking_health_df = pd.read_sql_query("SELECT * FROM tracking_health", conn)
conn.close()

if summary_df.empty:
    raise RuntimeError("summary table is empty. Rebuild analytics DB after running pipeline.")

eq_class_map = summary_df[["equipment_id", "equipment_class"]].drop_duplicates()
events_df = events_df.merge(eq_class_map, on="equipment_id", how="left")

class_options = ["ALL"] + sorted([x for x in summary_df["equipment_class"].dropna().unique().tolist() if str(x).strip()])
class_dd = widgets.Dropdown(options=class_options, value="ALL", description="Class:")
equipment_dd = widgets.Dropdown(options=["ALL"], value="ALL", description="Equipment:")
refresh_btn = widgets.Button(description="Refresh", button_style="primary")
out = widgets.Output()

def get_filtered_tables(class_value: str, equipment_value: str):
    s = summary_df.copy()
    a = activity_mix_df.copy()
    f = fragmentation_df.copy()
    e = events_df.copy()

    if class_value != "ALL":
        s = s[s["equipment_class"] == class_value]
        a = a[a["equipment_class"] == class_value]
        f = f[f["equipment_class"] == class_value]
        e = e[e["equipment_class"] == class_value]

    if equipment_value != "ALL":
        s = s[s["equipment_id"] == equipment_value]
        a = a[a["equipment_id"] == equipment_value]
        f = f[f["equipment_id"] == equipment_value]
        e = e[e["equipment_id"] == equipment_value]

    return s, a, f, e

def update_equipment_options(*_):
    class_value = class_dd.value
    if class_value == "ALL":
        opts = sorted(summary_df["equipment_id"].unique().tolist())
    else:
        opts = sorted(summary_df.loc[summary_df["equipment_class"] == class_value, "equipment_id"].unique().tolist())
    equipment_dd.options = ["ALL"] + opts
    if equipment_dd.value not in equipment_dd.options:
        equipment_dd.value = "ALL"

def render_dashboard(*_):
    class_value = class_dd.value
    equipment_value = equipment_dd.value
    s, a, f, e = get_filtered_tables(class_value, equipment_value)

    with out:
        clear_output(wait=True)
        display(Markdown(f"### Dashboard Scope: class={class_value}, equipment={equipment_value}"))

        if s.empty:
            display(Markdown("No rows for current filter."))
            return

        total_eq = int(s["equipment_id"].nunique())
        avg_util = float(s["utilization_percent"].mean())
        avg_tracks = float(f["tracks_per_equipment"].mean()) if not f.empty else 0.0
        max_tracks = int(f["tracks_per_equipment"].max()) if not f.empty else 0

        active_ratio = 0.0
        if not e.empty:
            active_ratio = float((e["state"] == "ACTIVE").mean() * 100.0)

        fig_cards = go.Figure()
        fig_cards.add_trace(go.Indicator(
            mode="number", value=total_eq, title={"text": "Equipment IDs"}, number={"font": {"size": 38}}, domain={"row": 0, "column": 0}
        ))
        fig_cards.add_trace(go.Indicator(
            mode="number", value=round(avg_util, 2), title={"text": "Avg Utilization %"}, number={"font": {"size": 38}, "suffix": "%"}, domain={"row": 0, "column": 1}
        ))
        fig_cards.add_trace(go.Indicator(
            mode="number", value=round(active_ratio, 2), title={"text": "Active Frame Ratio %"}, number={"font": {"size": 38}, "suffix": "%"}, domain={"row": 0, "column": 2}
        ))
        fig_cards.add_trace(go.Indicator(
            mode="number", value=max_tracks, title={"text": "Max Tracks/Equipment"}, number={"font": {"size": 38}}, domain={"row": 0, "column": 3}
        ))
        fig_cards.update_layout(grid={"rows": 1, "columns": 4, "pattern": "independent"}, height=180, margin={"l": 10, "r": 10, "t": 10, "b": 10})
        fig_cards.show()

        fig_util = px.bar(
            s.sort_values("utilization_percent", ascending=False),
            x="equipment_id",
            y="utilization_percent",
            color="equipment_class",
            title="Utilization Percent by Equipment",
            text="utilization_percent",
        )
        fig_util.update_traces(texttemplate="%{text:.2f}", textposition="outside")
        fig_util.update_layout(yaxis_title="Utilization %", xaxis_title="Equipment ID")
        fig_util.show()

        if not a.empty:
            activity_long = a.melt(
                id_vars=["equipment_id", "equipment_class"],
                value_vars=[
                    "digging_pct",
                    "swinging_loading_pct",
                    "pushing_pct",
                    "moving_pct",
                    "dumping_pct",
                    "waiting_pct",
                ],
                var_name="activity",
                value_name="percent",
            )
            activity_long["activity"] = activity_long["activity"].str.replace("_pct", "", regex=False).str.upper()
            fig_act = px.bar(
                activity_long,
                x="equipment_id",
                y="percent",
                color="activity",
                title="Activity Mix % by Equipment",
            )
            fig_act.update_layout(barmode="stack", yaxis_title="Percent")
            fig_act.show()

        if not f.empty:
            risk_order = {"low": 1, "medium": 2, "high": 3}
            f_plot = f.copy()
            f_plot["risk_rank"] = f_plot["fragmentation_risk"].map(risk_order).fillna(0)
            f_plot = f_plot.sort_values(["risk_rank", "tracks_per_equipment"], ascending=[False, False])
            fig_frag = px.bar(
                f_plot,
                x="equipment_id",
                y="tracks_per_equipment",
                color="fragmentation_risk",
                title="Fragmentation Audit (Tracks per Equipment)",
                color_discrete_map={"high": "#d62728", "medium": "#ff7f0e", "low": "#2ca02c"},
            )
            fig_frag.update_layout(yaxis_title="Distinct Track IDs")
            fig_frag.show()

        if not e.empty:
            trend = (
                e.groupby("frame_id", as_index=False)["state"]
                .apply(lambda x: (x == "ACTIVE").mean() * 100.0)
                .rename(columns={"state": "active_ratio_pct"})
            )
            fig_trend = px.line(
                trend,
                x="frame_id",
                y="active_ratio_pct",
                title="Active Ratio Trend Across Frames",
            )
            fig_trend.update_layout(yaxis_title="Active %", xaxis_title="Frame")
            fig_trend.show()

        display(Markdown("#### Tracking Health Snapshot"))
        display(tracking_health_df)

update_equipment_options()
class_dd.observe(update_equipment_options, names="value")
refresh_btn.on_click(render_dashboard)

controls = widgets.HBox([class_dd, equipment_dd, refresh_btn])
display(controls, out)
render_dashboard()

Output()

## Hardening + Deployment Bundle

In [41]:
# hardening audit and QA report generation
import json
import os
from datetime import datetime, timezone
import pandas as pd

OUT = globals().get("OUTPUT_DIR", "/content/outputs_baseline")
events_path = f"{OUT}/events.jsonl"
summary_path = f"{OUT}/summary.csv"
db_path = os.environ.get("ANALYTICS_DB", f"{OUT}/analytics_day3.db")
qa_json_path = f"{OUT}/qa_report.json"
qa_md_path = f"{OUT}/qa_report.md"

required_files = {
    "events_jsonl": events_path,
    "summary_csv": summary_path,
    "analytics_db": db_path,
    "run_meta_json": f"{OUT}/run_meta.json",
    "activity_mix_csv": f"{OUT}/activity_mix.csv",
    "fragmentation_csv": f"{OUT}/fragmentation_audit.csv",
    "tracking_health_csv": f"{OUT}/tracking_health.csv",
}

missing_files = []
file_sizes = {}
for name, path in required_files.items():
    if not os.path.exists(path):
        missing_files.append({"name": name, "path": path})
    else:
        file_sizes[name] = int(os.path.getsize(path))

if missing_files:
    raise FileNotFoundError(f"Missing required files: {missing_files}")

rows = []
with open(events_path, "r", encoding="utf-8") as f:
    for line in f:
        e = json.loads(line)
        util = e.get("utilization", {})
        ta = e.get("time_analytics", {})
        rows.append(
            {
                "frame_id": e.get("frame_id"),
                "equipment_id": e.get("equipment_id"),
                "track_id": e.get("track_id"),
                "equipment_class": e.get("equipment_class"),
                "state": util.get("current_state"),
                "activity": util.get("current_activity"),
                "motion_source": util.get("motion_source"),
                "total_tracked_seconds": ta.get("total_tracked_seconds", 0.0),
                "total_idle_seconds": ta.get("total_idle_seconds", 0.0),
                "utilization_percent": ta.get("utilization_percent", 0.0),
                "current_dwell_seconds": ta.get("current_dwell_seconds", 0.0),
            }
        )

events_df = pd.DataFrame(rows)
summary_df = pd.read_csv(summary_path)

if events_df.empty:
    raise RuntimeError("events.jsonl is empty. Run pipeline first.")

events_df = events_df.sort_values(["equipment_id", "frame_id"]).reset_index(drop=True)

active_waiting = int(((events_df["state"] == "ACTIVE") & (events_df["activity"] == "WAITING")).sum())
active_none = int(((events_df["state"] == "ACTIVE") & (events_df["motion_source"] == "none")).sum())
negative_dwell = int((events_df["current_dwell_seconds"] < 0).sum())
negative_tracked = int((events_df["total_tracked_seconds"] < 0).sum())
negative_idle = int((events_df["total_idle_seconds"] < 0).sum())

tracked_diff = events_df.groupby("equipment_id")["total_tracked_seconds"].diff()
idle_diff = events_df.groupby("equipment_id")["total_idle_seconds"].diff()
non_monotonic_tracked = int((tracked_diff < -1e-6).sum())
non_monotonic_idle = int((idle_diff < -1e-6).sum())

orphan_events = int((~events_df["equipment_id"].isin(summary_df["equipment_id"]) ).sum())
null_key_fields = int(
    events_df[["frame_id", "equipment_id", "state", "activity", "motion_source"]].isnull().any(axis=1).sum()
)

status = "PASS"
warnings = []
failures = []

if active_waiting > 0:
    warnings.append(f"ACTIVE+WAITING rows: {active_waiting}")
if active_none > 0:
    warnings.append(f"ACTIVE+none-motion rows: {active_none}")
if orphan_events > 0:
    warnings.append(f"Orphan events not present in summary: {orphan_events}")
if null_key_fields > 0:
    warnings.append(f"Rows with null key fields: {null_key_fields}")

if negative_dwell > 0:
    failures.append(f"Negative dwell rows: {negative_dwell}")
if negative_tracked > 0 or negative_idle > 0:
    failures.append(f"Negative cumulative times: tracked={negative_tracked}, idle={negative_idle}")
if non_monotonic_tracked > 0 or non_monotonic_idle > 0:
    failures.append(f"Non-monotonic cumulative times: tracked={non_monotonic_tracked}, idle={non_monotonic_idle}")

if failures:
    status = "FAIL"
elif warnings:
    status = "WARN"

metrics = {
    "total_events": int(len(events_df)),
    "summary_rows": int(len(summary_df)),
    "unique_equipment_ids": int(events_df["equipment_id"].nunique()),
    "active_frame_ratio_pct": round(float((events_df["state"] == "ACTIVE").mean() * 100.0), 2),
    "avg_utilization_pct": round(float(events_df["utilization_percent"].mean()), 2),
    "max_tracks_per_equipment": int(events_df.groupby("equipment_id")["track_id"].nunique().max()),
    "active_waiting_rows": active_waiting,
    "active_none_rows": active_none,
    "orphan_events": orphan_events,
    "null_key_field_rows": null_key_fields,
    "non_monotonic_tracked_rows": non_monotonic_tracked,
    "non_monotonic_idle_rows": non_monotonic_idle,
}

report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": status,
    "metrics": metrics,
    "warnings": warnings,
    "failures": failures,
    "file_sizes_bytes": file_sizes,
}

with open(qa_json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

md_lines = [
    "# QA Report",
    "",
    f"- status: {status}",
    f"- generated_at_utc: {report['generated_at_utc']}",
    "",
    "## Metrics",
]
for k, v in metrics.items():
    md_lines.append(f"- {k}: {v}")
if warnings:
    md_lines.append("")
    md_lines.append("## Warnings")
    for w in warnings:
        md_lines.append(f"- {w}")
if failures:
    md_lines.append("")
    md_lines.append("## Failures")
    for x in failures:
        md_lines.append(f"- {x}")

with open(qa_md_path, "w", encoding="utf-8") as f:
    f.write("\n".join(md_lines))

print("QA status:", status)
print("QA JSON:", qa_json_path)
print("QA Markdown:", qa_md_path)
print("Warnings:", warnings if warnings else "None")
print("Failures:", failures if failures else "None")

QA status: PASS
QA JSON: /content/outputs_baseline/qa_report.json
QA Markdown: /content/outputs_baseline/qa_report.md
Warnings: None
Failures: None


In [42]:
# build deploy-ready bundle with manifest
import hashlib
import json
import os
import shutil
from datetime import datetime, timezone

OUT = globals().get("OUTPUT_DIR", "/content/outputs_baseline")
deploy_dir = f"{OUT}/deploy_bundle"
bundle_zip = f"{OUT}/equipment_utilization_delivery.zip"

if os.path.exists(deploy_dir):
    shutil.rmtree(deploy_dir)
os.makedirs(deploy_dir, exist_ok=True)

candidate_files = [
    f"{OUT}/annotated.mp4",
    f"{OUT}/events.jsonl",
    f"{OUT}/summary.csv",
    f"{OUT}/analytics_day3.db",
    f"{OUT}/run_meta.json",
    f"{OUT}/qa_report.json",
    f"{OUT}/qa_report.md",
    f"{OUT}/activity_mix.csv",
    f"{OUT}/fragmentation_audit.csv",
    f"{OUT}/tracking_health.csv",
]

def sha256_of(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "bundle_name": os.path.basename(bundle_zip),
    "files": [],
}

copied_count = 0
for src in candidate_files:
    if not os.path.exists(src):
        continue
    name = os.path.basename(src)
    dst = os.path.join(deploy_dir, name)
    shutil.copy2(src, dst)
    copied_count += 1
    manifest["files"].append(
        {
            "name": name,
            "size_bytes": int(os.path.getsize(dst)),
            "sha256": sha256_of(dst),
        }
    )

readme_path = os.path.join(deploy_dir, "README_DEPLOY.md")
readme_text = "\n".join([
    "# Equipment Utilization Delivery Bundle",
    "",
    "## Contents",
    "- annotated.mp4: visual output",
    "- events.jsonl: frame-level events",
    "- summary.csv: per-equipment totals",
    "- analytics_day3.db: SQLite analytics database",
    "- activity_mix.csv / fragmentation_audit.csv / tracking_health.csv: dashboard-ready tables",
    "- qa_report.json / qa_report.md: integrity checks",
    "",
    "## Quick Validation",
    "1. Confirm qa_report.json status is PASS or WARN.",
    "2. Open tracking_health.csv and review max_tracks_per_equipment.",
    "3. Use analytics_day3.db in dashboard notebook cells.",
    "",
    "## Notes",
    "- Bundle is generated from current OUTPUT_DIR only.",
    "- Regenerate after each new pipeline run.",
])
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme_text)

manifest_path = os.path.join(deploy_dir, "manifest.json")
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

if os.path.exists(bundle_zip):
    os.remove(bundle_zip)
base_name = os.path.splitext(bundle_zip)[0]
shutil.make_archive(base_name, "zip", deploy_dir)

zip_size_mb = round(os.path.getsize(bundle_zip) / (1024 * 1024), 2) if os.path.exists(bundle_zip) else 0.0
print("Deploy folder:", deploy_dir)
print("Bundle zip:", bundle_zip)
print("Copied files:", copied_count)
print("Zip size (MB):", zip_size_mb)
print("Manifest entries:", len(manifest["files"]))

Deploy folder: /content/outputs_baseline/deploy_bundle
Bundle zip: /content/outputs_baseline/equipment_utilization_delivery.zip
Copied files: 10
Zip size (MB): 813.6
Manifest entries: 10


In [33]:
# Optional: zip outputs for one-click download
import shutil
shutil.make_archive('/content/outputs_baseline', 'zip', '/content/outputs_baseline')
print('Created:', '/content/outputs_baseline.zip')

Created: /content/outputs_baseline.zip
